# Unmix the Signal - results notebook
**1. Problem statement and objectives.** Recover independent sources from linear mixtures `X = A S` using centering, PCA whitening and symmetric FastICA, all implemented from scratch in `src/`. This notebook *imports* that implementation (no second copy of the algorithm) and analyses its outputs. Run `python main.py` first to create the files under `outputs/`.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd()/'config.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, matplotlib.pyplot as plt
import config as cfg
from src import audio_utils as au, mixing
from src.pipeline import separate
from src.whitening import covariance
from src.evaluation import evaluate_separation, correlation_matrix, amari_index
try:
    from IPython.display import Audio, display, Image
    HAVE_AUDIO = True
except Exception:
    HAVE_AUDIO = False   # fallback: open the WAV files in outputs/ with any audio player
meta = json.load(open(cfg.METRICS_DIR/'metrics_main.json'))
print('synthetic sources:', meta['synthetic_sources'])

**2. Dataset and source information.** The table shows where each source came from. If it says SYNTHETIC, the audio is generated by `src/synthetic_audio.py` (no real recordings were available offline).

In [ ]:
for s in meta['sources']: print(s)

**3. Loading and inspecting the clean audio** (re-created deterministically with the same seed as `main.py`).

In [ ]:
S, info = au.load_sources(cfg.SOURCES_DIR, cfg.SOURCE_NAMES, meta['sample_rate'], meta['duration_s'], meta['seed'])
print('S shape (n_sources, n_samples):', S.shape, '| means:', S.mean(1).round(12), '| stds:', S.std(1).round(6))
if HAVE_AUDIO:
    for nm in cfg.SOURCE_NAMES:
        p = cfg.SOURCES_DIR/f'synthetic_{nm}.wav'
        if p.exists(): print(nm); display(Audio(str(p)))

**4. Mixing matrix.** Known only because this is a controlled experiment; ICA never sees it.

In [ ]:
A = np.array(cfg.MIXING_MATRIX)
print(A); print(mixing.validate_mixing_matrix(A, 3))

**5. Mixtures** `X = A @ S`.

In [ ]:
X = mixing.mix_sources(S, A)
print('X shape:', X.shape)
if HAVE_AUDIO:
    for i in range(3): display(Audio(str(cfg.MIXTURES_DIR/f'mixture_{i+1}.wav')))

**6-8. Centering, PCA whitening, FastICA** through the shared pipeline function (`src/pipeline.py`).

In [ ]:
run = separate(X, seed=meta['seed'], eig_floor_rel=cfg.EIG_FLOOR_REL, **cfg.FASTICA)
print('max |mean(Xc)|:', np.abs(run.Xc.mean(1)).max())
print('eigenvalues:', run.whitening.eigvals)
print('Cov(Z) - I max abs:', np.abs(covariance(run.whitening.Z)-np.eye(3)).max())
print('FastICA converged:', run.ica.converged, 'iterations:', run.ica.n_iter)
fig, ax = plt.subplots(1,2, figsize=(9,3.5))
for a_, C, t in zip(ax, [covariance(run.Xc), covariance(run.whitening.Z)], ['Cov before whitening','Cov after whitening']):
    im = a_.imshow(C, cmap='coolwarm'); a_.set_title(t); a_.set_xlabel('channel'); a_.set_ylabel('channel'); fig.colorbar(im, ax=a_)
plt.show()

**9. Separated audio playback.** Each file is peak-normalised independently *for listening only*; this does not recover original amplitudes.

In [ ]:
if HAVE_AUDIO:
    for nm in cfg.SOURCE_NAMES:
        print(nm); display(Audio(str(cfg.SEPARATED_DIR/f'separated_{nm}.wav')))
else:
    print('Notebook audio widgets unavailable here; play the WAVs in', cfg.SEPARATED_DIR)

**10-11. Permutation matching, scale alignment and metrics.** Raw ICA outputs (`run.Y`) are never modified; evaluation uses aligned copies.

In [ ]:
ev = evaluate_separation(S, run.Y, cfg.SOURCE_NAMES)
print('correlation matrix (rows = sources, cols = raw components):'); print(ev['corr_matrix'].round(4))
print('assignment source->component:', ev['assignment'])
for p in ev['per_source']: print({k: (round(v,4) if isinstance(v,float) else v) for k,v in p.items()})
print('summary:', ev['summary']); print('Amari index:', amari_index(run.W_total @ A))
same = np.allclose(ev['summary']['mean_abs_pearson'], meta['summary']['mean_abs_pearson'])
print('Matches metrics_main.json written by main.py:', same)

**12. Visualisations** produced by `main.py` (`outputs/plots/`).

In [ ]:
if HAVE_AUDIO:
    for f in sorted(cfg.PLOTS_DIR.glob('*.png')): print(f.name); display(Image(str(f)))
else:
    print([f.name for f in sorted(cfg.PLOTS_DIR.glob('*.png'))])

**13. Discussion.** With synthetic sources that are independent *by construction* and strongly non-Gaussian, the ideal instantaneous model holds exactly, so FastICA recovers them almost perfectly (see metrics above). This is a best case. It says nothing about real speech/music/rain recordings, which are only approximately independent, and where the number of microphones, room acoustics and noise break the model. PCA/whitening alone do not separate the sources (Bonus A); noise degrades ICA gracefully but clearly (Bonus B).

**14. Limitations and future work.** Instantaneous linear mixing only (no delays/reverberation -> convolutive ICA needed); square case only; real recordings should be tested (place `speech.wav`, `music.wav`, `ambient.wav` in `data/sources/`); Gaussian-noise robustness needs noise-aware methods; SI-SDR here is a monotone function of correlation, so it is not independent evidence.